<a href="https://colab.research.google.com/github/krishnakanth-20/IPL-Cricket-Intelligence-RAG/blob/main/IPL_Cricket_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import sys

In [2]:
!pip -q install sentence-transformers
!pip -q install chromadb

## Load IPL Dataset from Google Drive

Mount Google Drive

In [3]:

from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Extract ZIP dataset

In [4]:

import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab_Notebooks/genai/datasets/ipl_male_csv2.zip"
extract_path = "/content/ipl_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path)[:10])


Dataset extracted successfully!
['1359536_info.csv', '829777_info.csv', '501240.csv', '734045.csv', '1529290.csv', '548377_info.csv', '392197_info.csv', '1216503_info.csv', '392186.csv', '1304110_info.csv']


Load dataset

In [5]:

df = pd.read_csv("/content/ipl_data/all_matches.csv",low_memory=False)

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())


Dataset Shape: (295732, 27)

Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import os
import shutil
import sqlite3
import zipfile
import chromadb

backup_path = "/content/drive/MyDrive/IPL_RAG"

# Restore SQLite database
os.makedirs("/content/ipl_database", exist_ok=True)

shutil.copy2(
    backup_path + "/ipl_stats.db",
    "/content/ipl_database/ipl_stats.db"
)

# Restore ChromaDB
os.makedirs("/content/ipl_chromadb", exist_ok=True)

with zipfile.ZipFile(
    backup_path + "/ipl_chromadb.zip", "r"
) as zip_ref:

    zip_ref.extractall("/content/ipl_chromadb")

# Connect SQLite
conn = sqlite3.connect(
    "/content/ipl_database/ipl_stats.db",
    check_same_thread=False
)

# Connect ChromaDB
chroma_client = chromadb.PersistentClient(
    path="/content/ipl_chromadb"
)

collection = chroma_client.get_collection(
    name="ipl_cricket"
)

print("SQLite Connected Successfully")
print("ChromaDB Documents:", collection.count())

SQLite Connected Successfully
ChromaDB Documents: 5162


## Data Cleaning & Preprocessing

In [6]:
# Dataset shape
print("Dataset Shape:", df.shape)

Dataset Shape: (295732, 27)


In [7]:
# Column names
print("\nColumn Names:")
print(df.columns.tolist())


Column Names:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']


In [8]:
# Data types
print("\nData Types:")
print(df.dtypes)


Data Types:
match_id                    int64
season                     object
start_date                 object
venue                      object
innings                     int64
ball                      float64
actual_delivery           float64
batting_team               object
bowling_team               object
striker                    object
non_striker                object
bowler                     object
runs_off_bat                int64
extras                      int64
wides                     float64
noballs                   float64
byes                      float64
legbyes                   float64
penalty                   float64
non_boundary               object
wicket_type                object
player_dismissed           object
other_wicket_type         float64
other_player_dismissed    float64
fielder_1                  object
fielder_2                  object
fielder_3                  object
dtype: object


In [9]:
# Missing values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
match_id                       0
season                         0
start_date                     0
venue                          0
innings                        0
ball                           0
actual_delivery                0
batting_team                   0
bowling_team                   0
striker                        0
non_striker                    0
bowler                         0
runs_off_bat                   0
extras                         0
wides                     285856
noballs                   294506
byes                      294987
legbyes                   291326
penalty                   295730
non_boundary              295703
wicket_type               281027
player_dismissed          281027
other_wicket_type         295732
other_player_dismissed    295732
fielder_1                 285011
fielder_2                 295200
fielder_3                 295719
dtype: int64


In [10]:
# Duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())


Duplicate Rows:
0


### Check Missing Values

In [11]:

# Check missing values
missing_values = df.isnull().sum()

# Missing percentage
missing_percentage = (
    df.isnull().mean() * 100
).round(2)

# Create summary
missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage
})

display(
    missing_summary.sort_values(
        by="Missing Count",
        ascending=False
    )
)


,Missing Count,Missing Percentage
other_player_dismissed,295732,100.00
other_wicket_type,295732,100.00
penalty,295730,100.00
fielder_3,295719,100.00
non_boundary,295703,99.99
fielder_2,295200,99.82
byes,294987,99.75
noballs,294506,99.59
legbyes,291326,98.51
wides,285856,96.66


Fill numerical extras columns

In [12]:

# Create a copy to preserve original data
df_clean = df.copy()

# Numerical columns
extra_columns = [
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty",
    "non_boundary"
]

# Replace missing values with zero
df_clean[extra_columns] = (
    df_clean[extra_columns].fillna(0)
)

print("Numerical missing values handled!")
print(df_clean[extra_columns].isnull().sum())


Numerical missing values handled!
wides           0
noballs         0
byes            0
legbyes         0
penalty         0
non_boundary    0
dtype: int64


Handle wicket and fielder columns

In [13]:

# Wicket information
df_clean["wicket_type"] = (
    df_clean["wicket_type"].fillna("No Wicket")
)

df_clean["player_dismissed"] = (
    df_clean["player_dismissed"].fillna("Not Out")
)

# Other wicket information
df_clean["other_wicket_type"] = (
    df_clean["other_wicket_type"].fillna("No Wicket")
)

df_clean["other_player_dismissed"] = (
    df_clean["other_player_dismissed"].fillna("Not Out")
)

# Fielder information
fielder_columns = [
    "fielder_1",
    "fielder_2",
    "fielder_3"
]

df_clean[fielder_columns] = (
    df_clean[fielder_columns].fillna("Not Applicable")
)

print("Wicket and fielder missing values handled!")


Wicket and fielder missing values handled!


Verify cleaning

In [14]:

print("Remaining Missing Values:")
print(df_clean.isnull().sum())

print("\nDataset Shape:")
print(df_clean.shape)

display(df_clean.head())


Remaining Missing Values:
match_id                  0
season                    0
start_date                0
venue                     0
innings                   0
ball                      0
actual_delivery           0
batting_team              0
bowling_team              0
striker                   0
non_striker               0
bowler                    0
runs_off_bat              0
extras                    0
wides                     0
noballs                   0
byes                      0
legbyes                   0
penalty                   0
non_boundary              0
wicket_type               0
player_dismissed          0
other_wicket_type         0
other_player_dismissed    0
fielder_1                 0
fielder_2                 0
fielder_3                 0
dtype: int64

Dataset Shape:
(295732, 27)


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,0.0,0.0,0,No Wicket,Not Out,No Wicket,Not Out,Not Applicable,Not Applicable,Not Applicable
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,0.0,0.0,0,No Wicket,Not Out,No Wicket,Not Out,Not Applicable,Not Applicable,Not Applicable
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,0.0,0.0,0,No Wicket,Not Out,No Wicket,Not Out,Not Applicable,Not Applicable,Not Applicable
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,0.0,0.0,0,No Wicket,Not Out,No Wicket,Not Out,Not Applicable,Not Applicable,Not Applicable
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,0.0,0.0,0,No Wicket,Not Out,No Wicket,Not Out,Not Applicable,Not Applicable,Not Applicable


### Data Type Conversion

In [15]:

print(df_clean.dtypes)


match_id                    int64
season                     object
start_date                 object
venue                      object
innings                     int64
ball                      float64
actual_delivery           float64
batting_team               object
bowling_team               object
striker                    object
non_striker                object
bowler                     object
runs_off_bat                int64
extras                      int64
wides                     float64
noballs                   float64
byes                      float64
legbyes                   float64
penalty                   float64
non_boundary               object
wicket_type                object
player_dismissed           object
other_wicket_type          object
other_player_dismissed     object
fielder_1                  object
fielder_2                  object
fielder_3                  object
dtype: object


In [16]:

df_clean["start_date"] = pd.to_datetime(
    df_clean["start_date"],
    errors="coerce"
)

print(df_clean["start_date"].dtype)
print(df_clean["start_date"].head())


datetime64[ns]
0   2017-04-05
1   2017-04-05
2   2017-04-05
3   2017-04-05
4   2017-04-05
Name: start_date, dtype: datetime64[ns]


In [17]:

numeric_columns = [
    "runs_off_bat",
    "extras",
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty",
    "non_boundary"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="raise"
    ).astype("int64")

print(df_clean[numeric_columns].dtypes)


runs_off_bat    int64
extras          int64
wides           int64
noballs         int64
byes            int64
legbyes         int64
penalty         int64
non_boundary    int64
dtype: object


### Feature Engineering

In [18]:

# 1. Total runs scored per delivery
df_clean["total_runs"] = (
    df_clean["runs_off_bat"] + df_clean["extras"]
)

# 2. Check whether a wicket occurred
df_clean["is_wicket"] = (
    (df_clean["wicket_type"] != "No Wicket") |
    (df_clean["other_wicket_type"] != "No Wicket")
).astype(int)

# 3. Check whether delivery is legal
df_clean["is_legal_delivery"] = (
    (df_clean["wides"] == 0) &
    (df_clean["noballs"] == 0)
).astype(int)

print("Feature Engineering Completed!")

display(
    df_clean[
        [
            "runs_off_bat",
            "extras",
            "total_runs",
            "wicket_type",
            "is_wicket",
            "is_legal_delivery"
        ]
    ].head(10)
)


Feature Engineering Completed!


,runs_off_bat,extras,total_runs,wicket_type,is_wicket,is_legal_delivery
0,0,0,0,No Wicket,0,1
1,0,0,0,No Wicket,0,1
2,4,0,4,No Wicket,0,1
3,0,0,0,No Wicket,0,1
4,0,2,2,No Wicket,0,0
5,0,0,0,No Wicket,0,1
6,0,1,1,No Wicket,0,1
7,1,0,1,No Wicket,0,1
8,4,0,4,No Wicket,0,1
9,0,1,1,No Wicket,0,0


In [19]:

print("Dataset Shape:", df_clean.shape)

print("\nTotal Runs:", df_clean["total_runs"].sum())

print("Total Dismissals:", df_clean["is_wicket"].sum())

print(
    "Total Legal Deliveries:",
    df_clean["is_legal_delivery"].sum()
)

print("\nNew Columns:")
print(df_clean.columns.tolist())


Dataset Shape: (295732, 30)

Total Runs: 401738
Total Dismissals: 14705
Total Legal Deliveries: 284630

New Columns:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'total_runs', 'is_wicket', 'is_legal_delivery']


### Batting Statistics Analysis

In [20]:

# Create batting statistics for each player
batting_stats = df_clean.groupby("striker").agg(
    total_runs=("runs_off_bat", "sum"),
    fours=("runs_off_bat", lambda x: (x == 4).sum()),
    sixes=("runs_off_bat", lambda x: (x == 6).sum())
).reset_index()

# Count balls faced (wides don't count)
balls_faced = (
    df_clean[df_clean["wides"] == 0]
    .groupby("striker")
    .size()
    .reset_index(name="balls_faced")
)

# Merge batting statistics with balls faced
batting_stats = batting_stats.merge(
    balls_faced,
    on="striker",
    how="left"
)

# Calculate batting strike rate
batting_stats["strike_rate"] = (
    batting_stats["total_runs"] /
    batting_stats["balls_faced"].replace(0, float("nan"))
) * 100

batting_stats["strike_rate"] = (
    batting_stats["strike_rate"].round(2)
)

# Sort by highest runs
batting_stats = batting_stats.sort_values(
    by="total_runs",
    ascending=False
)

display(batting_stats.head(10))


,striker,total_runs,fours,sixes,balls_faced,strike_rate
694,V Kohli,9346,847,317,6930,134.86
527,RG Sharma,7331,661,324,5517,132.88
564,S Dhawan,6769,768,153,5326,127.09
159,DA Warner,6567,663,236,4702,139.66
311,KL Rahul,5828,509,239,4188,139.16
598,SK Raina,5536,506,204,4046,136.83
409,MS Dhoni,5439,375,264,3957,137.45
52,AM Rahane,5367,540,141,4273,125.60
628,SV Samson,5181,432,243,3671,141.13
31,AB de Villiers,5181,414,253,3411,151.89


Strike Rate = (Total Runs / Balls Faced ) * 100

In [21]:

player = "V Kohli"

result = batting_stats[
    batting_stats["striker"].str.contains(
        player, case=False, na=False
    )
]

display(result)


,striker,total_runs,fours,sixes,balls_faced,strike_rate
694,V Kohli,9346,847,317,6930,134.86


### Calculate Bowling Statistics

In [22]:

# Dismissals credited to the bowler
bowler_wicket_types = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket"
]

# Create bowler wicket column
df_clean["bowler_wicket"] = (
    df_clean["wicket_type"]
    .str.lower()
    .isin(bowler_wicket_types)
    .astype(int)
)

# Include any second dismissal on the same delivery
df_clean["bowler_wicket"] += (
    df_clean["other_wicket_type"]
    .str.lower()
    .isin(bowler_wicket_types)
    .astype(int)
)

# Runs conceded by bowler
# Byes, legbyes and penalty runs are excluded
df_clean["runs_conceded"] = (
    df_clean["runs_off_bat"] +
    df_clean["wides"] +
    df_clean["noballs"]
)

# Aggregate bowling statistics
bowling_stats = df_clean.groupby("bowler").agg(
    total_wickets=("bowler_wicket", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    legal_balls=("is_legal_delivery", "sum")
).reset_index()

# Economy rate = Runs conceded / Overs bowled
bowling_stats["economy_rate"] = (
    bowling_stats["runs_conceded"] * 6 /
    bowling_stats["legal_balls"].replace(0, float("nan"))
).round(2)

# Sort by highest wickets
bowling_stats = bowling_stats.sort_values(
    by="total_wickets",
    ascending=False
)

display(bowling_stats.head(10))


,bowler,total_wickets,runs_conceded,legal_balls,economy_rate
570,YS Chahal,233,5417,4037,8.05
76,B Kumar,226,5913,4600,7.71
481,SP Narine,209,5273,4660,6.79
375,PP Chawla,192,5108,3850,7.96
214,JJ Bumrah,190,4469,3653,7.34
387,R Ashwin,187,5652,4710,7.20
131,DJ Bravo,183,4360,3120,8.38
406,RA Jadeja,180,5465,4254,7.71
433,Rashid Khan,179,4297,3543,7.28
8,A Mishra,174,4145,3371,7.38


Economy Rate = (Total Runs Conceded × 6) / Total Legal Balls Bowled

In [23]:

player = "JJ Bumrah"

result = bowling_stats[
    bowling_stats["bowler"].str.contains(
        player,
        case=False,
        na=False
    )
]

display(result)


,bowler,total_wickets,runs_conceded,legal_balls,economy_rate
214,JJ Bumrah,190,4469,3653,7.34


### Season-wise Batting Statistics

In [24]:

# Group by season and batsman
season_batting = df_clean.groupby(
    ["season", "striker"]
).agg(
    total_runs=("runs_off_bat", "sum"),
    fours=("runs_off_bat", lambda x: (x == 4).sum()),
    sixes=("runs_off_bat", lambda x: (x == 6).sum())
).reset_index()

# Calculate balls faced excluding wides
season_balls = (
    df_clean[df_clean["wides"] == 0]
    .groupby(["season", "striker"])
    .size()
    .reset_index(name="balls_faced")
)

# Merge
season_batting = season_batting.merge(
    season_balls,
    on=["season", "striker"],
    how="left"
)

# Calculate strike rate
season_batting["strike_rate"] = (
    season_batting["total_runs"] * 100 /
    season_batting["balls_faced"].replace(0, float("nan"))
).round(2)

print("Season-wise batting statistics created!")
display(season_batting.head())


Season-wise batting statistics created!


,season,striker,total_runs,fours,sixes,balls_faced,strike_rate
0,2007/08,A Chopra,42,5,0,52,80.77
1,2007/08,A Kumble,13,1,0,17,76.47
2,2007/08,A Mishra,37,3,0,42,88.10
3,2007/08,A Mukund,0,0,0,1,0.00
4,2007/08,A Nehra,3,0,0,13,23.08


In [25]:

top_2024 = season_batting[
    season_batting["season"].astype(str) == "2024"
].sort_values(
    by="total_runs",
    ascending=False
).head(10)

display(top_2024)


,season,striker,total_runs,fours,sixes,balls_faced,strike_rate
2606,2024,V Kohli,741,62,38,479,154.70
2559,2024,RD Gaikwad,583,58,18,413,141.16
2550,2024,R Parag,573,40,33,384,149.22
2601,2024,TM Head,567,64,32,296,191.55
2585,2024,SV Samson,531,48,24,346,153.47
2473,2024,B Sai Sudharsan,527,48,17,373,141.29
2507,2024,KL Rahul,520,45,19,382,136.13
2536,2024,N Pooran,499,35,36,280,178.21
2582,2024,SP Narine,488,50,33,270,180.74
2459,2024,Abhishek Sharma,484,36,42,237,204.22


In [26]:

print("Available IPL Seasons:")
print(sorted(df_clean["season"].astype(str).unique()))


Available IPL Seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


### Season-wise Bowling Statistics


In [27]:

# Group by season and bowler
season_bowling = df_clean.groupby(
    ["season", "bowler"]
).agg(
    total_wickets=("bowler_wicket", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    legal_balls=("is_legal_delivery", "sum")
).reset_index()

# Calculate economy rate
season_bowling["economy_rate"] = (
    season_bowling["runs_conceded"] * 6 /
    season_bowling["legal_balls"].replace(0, float("nan"))
).round(2)

# Calculate bowling average
season_bowling["bowling_average"] = (
    season_bowling["runs_conceded"] /
    season_bowling["total_wickets"].replace(0, float("nan"))
).round(2)

# Calculate bowling strike rate
season_bowling["bowling_strike_rate"] = (
    season_bowling["legal_balls"] /
    season_bowling["total_wickets"].replace(0, float("nan"))
).round(2)

print("Season-wise bowling statistics created!")

display(season_bowling.head(10))


Season-wise bowling statistics created!


,season,bowler,total_wickets,runs_conceded,legal_balls,economy_rate,bowling_average,bowling_strike_rate
0,2007/08,A Kumble,7,304,230,7.93,43.43,32.86
1,2007/08,A Mishra,11,138,119,6.96,12.55,10.82
2,2007/08,A Nehra,12,348,269,7.76,29.00,22.42
3,2007/08,A Nel,1,31,18,10.33,31.00,18.00
4,2007/08,A Symonds,0,101,41,14.78,NaN,NaN
5,2007/08,AA Noffke,1,40,24,10.00,40.00,24.00
6,2007/08,AB Agarkar,8,207,156,7.96,25.88,19.50
7,2007/08,AB Dinda,9,260,234,6.67,28.89,26.00
8,2007/08,AD Mascarenhas,2,29,24,7.25,14.50,12.00
9,2007/08,AM Nayar,1,68,42,9.71,68.00,42.00


In [28]:

top_bowlers_2024 = season_bowling[
    season_bowling["season"].astype(str) == "2024"
].sort_values(
    by="total_wickets",
    ascending=False
).head(10)

display(top_bowlers_2024)


,season,bowler,total_wickets,runs_conceded,legal_balls,economy_rate,bowling_average,bowling_strike_rate
1843,2024,HV Patel,24,477,294,9.73,19.88,12.25
1833,2024,CV Varun,21,402,300,8.04,19.14,14.29
1850,2024,JJ Bumrah,20,336,311,6.48,16.80,15.55
1829,2024,Avesh Khan,19,526,329,9.59,27.68,17.32
1845,2024,Harshit Rana,19,383,253,9.08,20.16,13.32
1827,2024,Arshdeep Singh,19,505,302,10.03,26.58,15.89
1816,2024,AD Russell,19,295,176,10.06,15.53,9.26
1931,2024,T Natarajan,19,465,308,9.06,24.47,16.21
1897,2024,PJ Cummins,18,566,366,9.28,31.44,20.33
1946,2024,YS Chahal,18,546,348,9.41,30.33,19.33


In [29]:

print("Total Season-Bowler Records:", len(season_bowling))

print("\nMissing Values:")
print(season_bowling.isnull().sum())

print("\nAvailable Seasons:")
print(sorted(season_bowling["season"].astype(str).unique()))


Total Season-Bowler Records: 2203

Missing Values:
season                   0
bowler                   0
total_wickets            0
runs_conceded            0
legal_balls              0
economy_rate             0
bowling_average        391
bowling_strike_rate    391
dtype: int64

Available Seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


## SQL Database Preparation


### Create SQLite Database


In [30]:

import sqlite3

# Create database folder
os.makedirs("/content/ipl_database", exist_ok=True)

# Database path
db_path = "/content/ipl_database/ipl_stats.db"

# Connect to SQLite
conn = sqlite3.connect(db_path)

print("SQLite Database Connected Successfully!")


SQLite Database Connected Successfully!


In [31]:

# Store batting statistics
batting_stats.to_sql(
    "batting_stats",
    conn,
    if_exists="replace",
    index=False
)

# Store bowling statistics
bowling_stats.to_sql(
    "bowling_stats",
    conn,
    if_exists="replace",
    index=False
)

# Store season-wise batting statistics
season_batting.to_sql(
    "season_batting",
    conn,
    if_exists="replace",
    index=False
)

# Store season-wise bowling statistics
season_bowling.to_sql(
    "season_bowling",
    conn,
    if_exists="replace",
    index=False
)

conn.commit()

print("All IPL statistics stored successfully!")


All IPL statistics stored successfully!


In [32]:

query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)

display(tables)


,name
0,batting_stats
1,bowling_stats
2,season_batting
3,season_bowling


Save database to Google Drive

In [33]:

import shutil

conn.commit()
conn.close()

drive_folder = "/content/drive/MyDrive/IPL_RAG"
os.makedirs(drive_folder, exist_ok=True)

shutil.copy2(
    db_path,
    f"{drive_folder}/ipl_stats.db"
)

print("Database saved to Google Drive!")


Database saved to Google Drive!


Retrieve Cricket Statistics Using SQL

In [34]:

import sqlite3

db_path = "/content/ipl_database/ipl_stats.db"

conn = sqlite3.connect(db_path)

print("Database connected successfully!")


Database connected successfully!


Find top 10 run scorers in IPL 2024

In [35]:

query = """
SELECT
    striker,
    total_runs,
    balls_faced,
    strike_rate
FROM season_batting
WHERE CAST(season AS TEXT) = '2024'
ORDER BY total_runs DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)


,striker,total_runs,balls_faced,strike_rate
0,V Kohli,741,479,154.70
1,RD Gaikwad,583,413,141.16
2,R Parag,573,384,149.22
3,TM Head,567,296,191.55
4,SV Samson,531,346,153.47
5,B Sai Sudharsan,527,373,141.29
6,KL Rahul,520,382,136.13
7,N Pooran,499,280,178.21
8,SP Narine,488,270,180.74
9,Abhishek Sharma,484,237,204.22


Find top 10 wicket takers in IPL 2024

In [36]:

query = """
SELECT
    bowler,
    total_wickets,
    economy_rate,
    bowling_average
FROM season_bowling
WHERE CAST(season AS TEXT) = '2024'
ORDER BY total_wickets DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)


,bowler,total_wickets,economy_rate,bowling_average
0,HV Patel,24,9.73,19.88
1,CV Varun,21,8.04,19.14
2,JJ Bumrah,20,6.48,16.80
3,AD Russell,19,10.06,15.53
4,Arshdeep Singh,19,10.03,26.58
5,Avesh Khan,19,9.59,27.68
6,Harshit Rana,19,9.08,20.16
7,T Natarajan,19,9.06,24.47
8,PJ Cummins,18,9.28,31.44
9,YS Chahal,18,9.41,30.33


## Prepare Documents for Vector Database

Create Batting Documents

In [37]:

# Convert season-wise batting statistics into documents

batting_documents = []

for _, row in season_batting.iterrows():

    document = (
        f"In IPL season {row['season']}, "
        f"cricketer {row['striker']} scored "
        f"{row['total_runs']} runs from "
        f"{row['balls_faced']} balls. "
        f"The batting strike rate was "
        f"{row['strike_rate']}. "
        f"The player hit {row['fours']} fours "
        f"and {row['sixes']} sixes."
    )

    batting_documents.append(document)

print("Total Batting Documents:", len(batting_documents))

print("\nSample Document:")
print(batting_documents[0])


Total Batting Documents: 2959

Sample Document:
In IPL season 2007/08, cricketer A Chopra scored 42 runs from 52 balls. The batting strike rate was 80.77. The player hit 5 fours and 0 sixes.


In [38]:

print("Number of Documents:", len(batting_documents))

print("\nFirst 3 Documents:\n")

for doc in batting_documents[:3]:
    print(doc)
    print("-" * 60)


Number of Documents: 2959

First 3 Documents:

In IPL season 2007/08, cricketer A Chopra scored 42 runs from 52 balls. The batting strike rate was 80.77. The player hit 5 fours and 0 sixes.
------------------------------------------------------------
In IPL season 2007/08, cricketer A Kumble scored 13 runs from 17 balls. The batting strike rate was 76.47. The player hit 1 fours and 0 sixes.
------------------------------------------------------------
In IPL season 2007/08, cricketer A Mishra scored 37 runs from 42 balls. The batting strike rate was 88.1. The player hit 3 fours and 0 sixes.
------------------------------------------------------------


Create Bowling Performance Documents

In [39]:

# Create bowling performance documents

bowling_documents = []

for _, row in season_bowling.iterrows():

    document = (
        f"In IPL season {row['season']}, "
        f"bowler {row['bowler']} took "
        f"{row['total_wickets']} wickets. "
        f"The bowler conceded {row['runs_conceded']} runs "
        f"in {row['legal_balls']} legal deliveries. "
        f"The economy rate was {row['economy_rate']}. "
        f"The bowling average was {row['bowling_average']} "
        f"and the bowling strike rate was "
        f"{row['bowling_strike_rate']}."
    )

    bowling_documents.append(document)

print("Total Bowling Documents:", len(bowling_documents))

print("\nSample Bowling Document:")
print(bowling_documents[0])


Total Bowling Documents: 2203

Sample Bowling Document:
In IPL season 2007/08, bowler A Kumble took 7 wickets. The bowler conceded 304 runs in 230 legal deliveries. The economy rate was 7.93. The bowling average was 43.43 and the bowling strike rate was 32.86.


In [40]:

print("Total Bowling Documents:", len(bowling_documents))

print("\nFirst 3 Bowling Documents:\n")

for doc in bowling_documents[:3]:
    print(doc)
    print("-" * 60)


Total Bowling Documents: 2203

First 3 Bowling Documents:

In IPL season 2007/08, bowler A Kumble took 7 wickets. The bowler conceded 304 runs in 230 legal deliveries. The economy rate was 7.93. The bowling average was 43.43 and the bowling strike rate was 32.86.
------------------------------------------------------------
In IPL season 2007/08, bowler A Mishra took 11 wickets. The bowler conceded 138 runs in 119 legal deliveries. The economy rate was 6.96. The bowling average was 12.55 and the bowling strike rate was 10.82.
------------------------------------------------------------
In IPL season 2007/08, bowler A Nehra took 12 wickets. The bowler conceded 348 runs in 269 legal deliveries. The economy rate was 7.76. The bowling average was 29.0 and the bowling strike rate was 22.42.
------------------------------------------------------------


Combine batting and bowling documents

In [41]:

all_documents = batting_documents + bowling_documents

print("Batting Documents:", len(batting_documents))
print("Bowling Documents:", len(bowling_documents))
print("Total RAG Documents:", len(all_documents))

print("\nSample Combined Document:")
print(all_documents[0])


Batting Documents: 2959
Bowling Documents: 2203
Total RAG Documents: 5162

Sample Combined Document:
In IPL season 2007/08, cricketer A Chopra scored 42 runs from 52 balls. The batting strike rate was 80.77. The player hit 5 fours and 0 sixes.


## Generate Text Embeddings using Sentence Transformers

Embedding:
  * An embedding is a numerical representation of text that helps a computer compare its meaning with other text.

### Load Embedding Model

In [4]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded successfully!")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded successfully!


### Generate Embeddings

In [43]:

all_embeddings = model.encode(
    all_documents,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Total Documents:", len(all_documents))
print("Embedding Shape:", all_embeddings.shape)


Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Total Documents: 5162
Embedding Shape: (5162, 768)


### Store Embeddings in ChromaDB

`ChromaDB` is a vector database used to store and retrieve embeddings based on semantic similarity.

In [44]:

import chromadb
import os

# Create database folder
chroma_path = "/content/ipl_chromadb"

os.makedirs(chroma_path, exist_ok=True)

# Create persistent client
chroma_client = chromadb.PersistentClient(
    path=chroma_path
)

# Create collection
collection = chroma_client.get_or_create_collection(
    name="ipl_cricket",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")


ChromaDB collection created successfully!


Create document IDs and metadata

In [45]:

document_ids = []
metadatas = []

# Batting metadata
for i, (_, row) in enumerate(
    season_batting.iterrows()
):
    document_ids.append(f"batting_{i}")

    metadatas.append({
        "player": str(row["striker"]),
        "season": str(row["season"]),
        "type": "batting"
    })

# Bowling metadata
for i, (_, row) in enumerate(
    season_bowling.iterrows()
):
    document_ids.append(f"bowling_{i}")

    metadatas.append({
        "player": str(row["bowler"]),
        "season": str(row["season"]),
        "type": "bowling"
    })

print("Total IDs:", len(document_ids))
print("Total Metadata:", len(metadatas))

print("\nSample Metadata:")
print(metadatas[0])


Total IDs: 5162
Total Metadata: 5162

Sample Metadata:
{'player': 'A Chopra', 'season': '2007/08', 'type': 'batting'}


Store documents and embeddings

In [46]:

# Verify all lengths match
assert (
    len(all_documents)
    == len(all_embeddings)
    == len(document_ids)
    == len(metadatas)
)

# Insert documents in batches
batch_size = 500

for i in range(0, len(all_documents), batch_size):

    end = i + batch_size

    collection.upsert(
        ids=document_ids[i:end],
        documents=all_documents[i:end],
        embeddings=all_embeddings[i:end].tolist(),
        metadatas=metadatas[i:end]
    )

print("All documents stored successfully!")
print("Total Stored Documents:", collection.count())


All documents stored successfully!
Total Stored Documents: 5162


In [47]:

result = collection.peek(limit=2)

for i in range(len(result["ids"])):

    print("ID:", result["ids"][i])

    print("Document:", result["documents"][i])

    print("Metadata:", result["metadatas"][i])

    print("-" * 60)

print("Total Documents:", collection.count())


ID: batting_0
Document: In IPL season 2007/08, cricketer A Chopra scored 42 runs from 52 balls. The batting strike rate was 80.77. The player hit 5 fours and 0 sixes.
Metadata: {'season': '2007/08', 'type': 'batting', 'player': 'A Chopra'}
------------------------------------------------------------
ID: batting_1
Document: In IPL season 2007/08, cricketer A Kumble scored 13 runs from 17 balls. The batting strike rate was 76.47. The player hit 1 fours and 0 sixes.
Metadata: {'season': '2007/08', 'player': 'A Kumble', 'type': 'batting'}
------------------------------------------------------------
Total Documents: 5162


## Semantic Search & Retriever Development

Generate query embedding

In [52]:
question = "Explain Virat Kohli's batting performance in 2024"

query_embedding = model.encode(
    "Represent this sentence for searching relevant passages: "
    + question,
    normalize_embeddings=True
)

Search relevant documents

In [53]:

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i in range(len(results["documents"][0])):

    print(f"Document {i + 1}")

    print("Content:", results["documents"][0][i])

    print("Metadata:", results["metadatas"][0][i])

    print("Distance:", results["distances"][0][i])

    print("-" * 80)


Document 1
Content: In IPL season 2024, cricketer V Kohli scored 741 runs from 479 balls. The batting strike rate was 154.7. The player hit 62 fours and 38 sixes.
Metadata: {'season': '2024', 'player': 'V Kohli', 'type': 'batting'}
Distance: 0.2555915117263794
--------------------------------------------------------------------------------
Document 2
Content: In IPL season 2025, cricketer V Kohli scored 657 runs from 454 balls. The batting strike rate was 144.71. The player hit 66 fours and 19 sixes.
Metadata: {'season': '2025', 'type': 'batting', 'player': 'V Kohli'}
Distance: 0.28633683919906616
--------------------------------------------------------------------------------
Document 3
Content: In IPL season 2021, cricketer V Kohli scored 405 runs from 339 balls. The batting strike rate was 119.47. The player hit 43 fours and 9 sixes.
Metadata: {'season': '2021', 'type': 'batting', 'player': 'V Kohli'}
Distance: 0.29441213607788086
----------------------------------------------------

Create reusable retriever function

In [54]:

def retrieve_documents(question, top_k=5):

    query_embedding = model.encode(
        "Represent this sentence for searching relevant passages: "
        + question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    return results


Example

In [55]:

question = "Explain Virat Kohli's batting performance in 2024"

results = retrieve_documents(question, top_k=5)

for i, doc in enumerate(results["documents"][0]):

    print(f"Document {i + 1}:")
    print(doc)
    print("-" * 80)


Document 1:
In IPL season 2024, cricketer V Kohli scored 741 runs from 479 balls. The batting strike rate was 154.7. The player hit 62 fours and 38 sixes.
--------------------------------------------------------------------------------
Document 2:
In IPL season 2025, cricketer V Kohli scored 657 runs from 454 balls. The batting strike rate was 144.71. The player hit 66 fours and 19 sixes.
--------------------------------------------------------------------------------
Document 3:
In IPL season 2021, cricketer V Kohli scored 405 runs from 339 balls. The batting strike rate was 119.47. The player hit 43 fours and 9 sixes.
--------------------------------------------------------------------------------
Document 4:
In IPL season 2022, cricketer V Kohli scored 341 runs from 294 balls. The batting strike rate was 115.99. The player hit 32 fours and 8 sixes.
--------------------------------------------------------------------------------
Document 5:
In IPL season 2024, cricketer Yudhvir Singh

### Metadata Filtering

In [56]:

def advanced_retriever(
    question,
    player=None,
    season=None,
    doc_type=None,
    top_k=5
):

    filters = []

    if player is not None:
        filters.append({
            "player": {"$eq": player}
        })

    if season is not None:
        filters.append({
            "season": {"$eq": str(season)}
        })

    if doc_type is not None:
        filters.append({
            "type": {"$eq": doc_type}
        })

    # Prepare metadata filter
    where_filter = None

    if len(filters) == 1:
        where_filter = filters[0]

    elif len(filters) > 1:
        where_filter = {"$and": filters}

    # Generate query embedding
    query_embedding = model.encode(
        "Represent this sentence for searching relevant passages: "
        + question,
        normalize_embeddings=True
    )

    # Prepare query parameters
    query_params = {
        "query_embeddings": [
            query_embedding.tolist()
        ],
        "n_results": top_k,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if where_filter is not None:
        query_params["where"] = where_filter

    # Retrieve documents
    results = collection.query(**query_params)

    return results


Eample


In [57]:

results = advanced_retriever(
    question="Explain Bumrah bowling performance",
    player="JJ Bumrah",
    season=2024,
    doc_type="bowling",
    top_k=3
)

for i, doc in enumerate(results["documents"][0]):

    print("Document:", i + 1)
    print(doc)

    print("Metadata:")
    print(results["metadatas"][0][i])

    print("-" * 60)

if not results["documents"][0]:
    print("No matching documents found.")


Document: 1
In IPL season 2024, bowler JJ Bumrah took 20 wickets. The bowler conceded 336 runs in 311 legal deliveries. The economy rate was 6.48. The bowling average was 16.8 and the bowling strike rate was 15.55.
Metadata:
{'season': '2024', 'type': 'bowling', 'player': 'JJ Bumrah'}
------------------------------------------------------------


### Query Understanding

Create player name mapping

In [58]:

player_mapping = {
    "virat kohli": "V Kohli",
    "kohli": "V Kohli",
    "rohit sharma": "RG Sharma",
    "rohit": "RG Sharma",
    "ms dhoni": "MS Dhoni",
    "dhoni": "MS Dhoni",
    "jasprit bumrah": "JJ Bumrah",
    "bumrah": "JJ Bumrah",
    "ravindra jadeja": "RA Jadeja",
    "jadeja": "RA Jadeja",
    "hardik pandya": "HH Pandya",
    "rashid khan": "Rashid Khan",
    "david warner": "DA Warner",
    "chris gayle": "CH Gayle",
    "ab de villiers": "AB de Villiers"
}

print("Player mapping created")


Player mapping created


Extract player name

In [59]:

import re

def extract_player(question):

    question = question.lower()

    for name in sorted(
        player_mapping,
        key=len,
        reverse=True
    ):

        if re.search(
            r"\b" + re.escape(name) + r"\b",
            question
        ):
            return player_mapping[name]

    return None


print(extract_player(
    "How many runs did Virat Kohli score?"
))


V Kohli


Extract IPL season

In [60]:

def extract_season(question):

    match = re.search(
        r"\b20\d{2}\b",
        question
    )

    if match:
        return match.group()

    return None


In [61]:
print(extract_season(
    "Kohli performance in IPL 2024"
))

2024


Identify batting or bowling category

In [62]:

def extract_category(question):

    question = question.lower()

    bowling_keywords = [
        "bowling", "bowler", "wickets",
        "wicket", "economy", "bowled"
    ]

    batting_keywords = [
        "batting", "batsman", "batter",
        "runs", "sixes", "fours",
        "scored", "boundaries"
    ]

    if any(word in question
           for word in bowling_keywords):
        return "bowling"

    if any(word in question
           for word in batting_keywords):
        return "batting"

    return None


In [63]:
print(extract_category(
    "How many wickets did Bumrah take?"
))

bowling


Intent Classification

In [64]:

def classify_intent(question):

    question = question.lower()

    sql_keywords = [
        "how many", "total", "highest",
        "lowest", "most", "least",
        "top", "rank", "average",
        "count", "strike rate", "economy"
    ]

    vector_keywords = [
        "explain", "describe",
        "summarize", "analysis",
        "performance", "why"
    ]

    sql_match = any(
        word in question for word in sql_keywords
    )

    vector_match = any(
        word in question for word in vector_keywords
    )

    if "compare" in question:
        return "hybrid"

    if sql_match and vector_match:
        return "hybrid"

    if sql_match:
        return "sql"

    if vector_match:
        return "vector"

    return "hybrid"


Combine all functions

In [65]:

def understand_query(question):

    return {
        "question": question,
        "player": extract_player(question),
        "season": extract_season(question),
        "category": extract_category(question),
        "intent": classify_intent(question)
    }


result = understand_query(
    "How many runs did Virat Kohli score in IPL 2024?"
)

print(result)


{'question': 'How many runs did Virat Kohli score in IPL 2024?', 'player': 'V Kohli', 'season': '2024', 'category': 'batting', 'intent': 'sql'}


Test Question

In [66]:

questions = [
    "How many runs did Kohli score in 2024?",
    "Explain Bumrah bowling performance in 2023",
    "Who scored the most runs in 2024?",
    "Compare Kohli and Rohit batting performance"
]

for question in questions:

    result = understand_query(question)

    for key, value in result.items():
        print(f"{key}: {value}")

    print("-" * 60)


question: How many runs did Kohli score in 2024?
player: V Kohli
season: 2024
category: batting
intent: sql
------------------------------------------------------------
question: Explain Bumrah bowling performance in 2023
player: JJ Bumrah
season: 2023
category: bowling
intent: vector
------------------------------------------------------------
question: Who scored the most runs in 2024?
player: None
season: 2024
category: batting
intent: sql
------------------------------------------------------------
question: Compare Kohli and Rohit batting performance
player: V Kohli
season: None
category: batting
intent: hybrid
------------------------------------------------------------


## SQL Retriever Development

### Connect SQLite database

In [67]:

db_path = "/content/ipl_database/ipl_stats.db"

if not os.path.exists(db_path):
    db_path = "/content/drive/MyDrive/IPL_RAG/ipl_stats.db"

if not os.path.exists(db_path):
    raise FileNotFoundError("IPL database not found")

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

print(tables)


             name
0   batting_stats
1   bowling_stats
2  season_batting
3  season_bowling


### Create batting retriever

In [68]:

def get_batting_stats(
    player=None,
    season=None,
    limit=10
):

    query = """
        SELECT *
        FROM season_batting
        WHERE 1=1
    """

    params = []

    if player is not None:
        query += " AND striker = ?"
        params.append(player)

    if season is not None:
        query += " AND season = ?"
        params.append(str(season))

    query += " ORDER BY total_runs DESC LIMIT ?"
    params.append(limit)

    result = pd.read_sql_query(
        query,
        conn,
        params=params
    )

    return result


### Create bowling retriever

In [69]:

def get_bowling_stats(
    player=None,
    season=None,
    limit=10
):

    query = """
        SELECT *
        FROM season_bowling
        WHERE 1=1
    """

    params = []

    if player is not None:
        query += " AND bowler = ?"
        params.append(player)

    if season is not None:
        query += " AND season = ?"
        params.append(str(season))

    query += " ORDER BY total_wickets DESC LIMIT ?"
    params.append(limit)

    result = pd.read_sql_query(
        query,
        conn,
        params=params
    )

    return result


### Create automatic SQL Retriever

In [70]:

def sql_retriever(question):

    query_info = understand_query(question)

    player = query_info["player"]
    season = query_info["season"]
    category = query_info["category"]

    print("Extracted Information:", query_info)

    if category == "batting":

        return get_batting_stats(
            player=player,
            season=season
        )

    elif category == "bowling":

        return get_bowling_stats(
            player=player,
            season=season
        )

    else:

        return {
            "batting": get_batting_stats(
                player=player,
                season=season
            ),

            "bowling": get_bowling_stats(
                player=player,
                season=season
            )
        }


Example

In [71]:
sql_retriever(
    "How many runs did dhoni score in IPL 2024?"
)

Extracted Information: {'question': 'How many runs did dhoni score in IPL 2024?', 'player': 'MS Dhoni', 'season': '2024', 'category': 'batting', 'intent': 'sql'}


,season,striker,total_runs,fours,sixes,balls_faced,strike_rate
0,2024,MS Dhoni,161,14,13,73,220.55


In [72]:
sql_retriever(
    "Who scored the most runs in IPL 2023?"
)

Extracted Information: {'question': 'Who scored the most runs in IPL 2023?', 'player': None, 'season': '2023', 'category': 'batting', 'intent': 'sql'}


,season,striker,total_runs,fours,sixes,balls_faced,strike_rate
0,2023,Shubman Gill,890,85,33,564,157.80
1,2023,F du Plessis,730,60,36,475,153.68
2,2023,DP Conway,672,77,18,481,139.71
3,2023,V Kohli,639,65,16,457,139.82
4,2023,YBK Jaiswal,625,83,26,382,163.61
5,2023,SA Yadav,605,65,28,334,181.14
6,2023,RD Gaikwad,590,46,30,400,147.50
7,2023,DA Warner,516,69,10,392,131.63
8,2023,RK Singh,474,31,29,317,149.53
9,2023,Ishan Kishan,454,54,18,318,142.77


## Hybrid Retrieval

###  Create vector retrieval function

In [73]:

def vector_retriever(question, top_k=5):

    query_info = understand_query(question)

    results = advanced_retriever(
        question=question,
        player=query_info["player"],
        season=query_info["season"],
        doc_type=query_info["category"],
        top_k=top_k
    )

    return results["documents"][0]


### Convert SQL results into text

In [74]:

def format_sql_results(results):

    if isinstance(results, pd.DataFrame):

        if results.empty:
            return "No matching SQL records found."

        return results.to_string(index=False)

    elif isinstance(results, dict):

        context = []

        for category, data in results.items():

            context.append(
                f"{category.upper()} STATISTICS:\n"
                + format_sql_results(data)
            )

        return "\n\n".join(context)

    return str(results)


### Create Hybrid Retriever

In [75]:

def hybrid_retriever(question, top_k=5):

    query_info = understand_query(question)

    intent = query_info["intent"]

    sql_context = ""
    vector_context = ""

    # SQL Retrieval
    if intent in ["sql", "hybrid"]:

        sql_results = sql_retriever(question)

        sql_context = format_sql_results(
            sql_results
        )

    # Vector Retrieval
    if intent in ["vector", "hybrid"]:

        vector_results = vector_retriever(
            question,
            top_k=top_k
        )

        vector_context = "\n\n".join(
            vector_results
        )

    # Combine retrieved context
    combined_context = []

    if sql_context:
        combined_context.append(
            "SQL STATISTICS:\n" + sql_context
        )

    if vector_context:
        combined_context.append(
            "VECTOR DOCUMENTS:\n" + vector_context
        )

    final_context = "\n\n".join(
        combined_context
    )

    return {
        "question": question,
        "query_info": query_info,
        "intent": intent,
        "sql_context": sql_context,
        "vector_context": vector_context,
        "final_context": final_context
    }


SQL Output

In [76]:
result = hybrid_retriever(
    "How many runs did Kohli score in 2024?"
)

print("Intent:", result["intent"])
print(result["final_context"])

Extracted Information: {'question': 'How many runs did Kohli score in 2024?', 'player': 'V Kohli', 'season': '2024', 'category': 'batting', 'intent': 'sql'}
Intent: sql
SQL STATISTICS:
season striker  total_runs  fours  sixes  balls_faced  strike_rate
  2024 V Kohli         741     62     38          479        154.7


Vector Output

In [77]:
result = hybrid_retriever(
    "Explain Bumrah bowling performance in 2024"
)

print("Intent:", result["intent"])
print(result["final_context"])

Intent: vector
VECTOR DOCUMENTS:
In IPL season 2024, bowler JJ Bumrah took 20 wickets. The bowler conceded 336 runs in 311 legal deliveries. The economy rate was 6.48. The bowling average was 16.8 and the bowling strike rate was 15.55.


Hybrid Output

In [78]:
result = hybrid_retriever(
    "Explain Kohli batting performance and total runs in 2024"
)

print("Intent:", result["intent"])
print(result["final_context"])

Extracted Information: {'question': 'Explain Kohli batting performance and total runs in 2024', 'player': 'V Kohli', 'season': '2024', 'category': 'batting', 'intent': 'hybrid'}
Intent: hybrid
SQL STATISTICS:
season striker  total_runs  fours  sixes  balls_faced  strike_rate
  2024 V Kohli         741     62     38          479        154.7

VECTOR DOCUMENTS:
In IPL season 2024, cricketer V Kohli scored 741 runs from 479 balls. The batting strike rate was 154.7. The player hit 62 fours and 38 sixes.


## Gemini LLM Integration

In [79]:
!pip -q install -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [80]:

from google import genai
from google.colab import userdata

api_key = userdata.get("IPLKEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized")


Gemini client initialized


## RAG Prompt Engineering

In [82]:
MODEL_NAME = "gemini-2.5-flash"

def create_prompt(question, context):
    return f"""
You are an expert IPL cricket data analyst.

Instructions:
1. Answer using only the provided context.
2. Do not invent cricket statistics.
3. Prioritize SQL data for numerical answers.
4. Use vector documents for contextual explanations.
5. If information is insufficient, clearly mention it.
6. Provide clear and concise answers.
7. Do not assume missing player or season information.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""

### Hybrid RAG Pipeline

In [83]:
from google.genai import types, errors

# Fallbacks to robustly supported models in your environment
FALLBACK_MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.5-flash"
]

def generate_with_fallback(prompt):
    for model_name in FALLBACK_MODELS:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1
                )
            )
            if response.text:
                return {
                    "answer": response.text,
                    "model": model_name
                }
        except Exception as e:
            print(f"{model_name} unavailable: {e}. Trying next model...")
            continue

    return {
        "answer": "AI service temporarily unavailable.",
        "model": None
    }

In [84]:

def generate_rag_response(question, top_k=5):

    # Retrieve relevant context
    retrieval_result = hybrid_retriever(
        question,
        top_k=top_k
    )

    context = retrieval_result["final_context"]

    # Check context
    if not context.strip():
        return {
            "question": question,
            "answer": "No relevant IPL data found.",
            "intent": retrieval_result["intent"],
            "model": None,
            "context": context
        }

    # Create prompt
    prompt = create_prompt(question, context)

    # Generate answer with fallback
    response = generate_with_fallback(prompt)

    return {
        "question": question,
        "answer": response["answer"],
        "intent": retrieval_result["intent"],
        "model": response["model"],
        "context": context
    }


In [85]:
questions = [
    "How many runs did Kohli score in 2024?",
    "Explain Bumrah bowling performance in 2024",
    "Explain Kohli batting performance and total runs in 2024",
    "Who scored the most runs in IPL 2024?"
]

for question in questions:

    result = generate_rag_response(question)

    print("Question:", question)
    print("Intent:", result["intent"])
    print("Model:", result["model"])
    print("Answer:", result["answer"])
    print("-" * 80)

Extracted Information: {'question': 'How many runs did Kohli score in 2024?', 'player': 'V Kohli', 'season': '2024', 'category': 'batting', 'intent': 'sql'}
gemini-2.5-flash unavailable: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}. Trying next model...
gemini-2.5-pro unavailable: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use models/gemini-3.1-pro-preview for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}. Trying next model...
Question: How many runs did Kohli score in 2024?
Intent: sq

## RAG Evaluation

In [86]:

import pandas as pd
import time
import re

# Evaluation dataset
test_data = [
    ("How many runs did Kohli score in IPL 2022?",
     "341", "sql"),

    ("How many wickets did Bumrah take in IPL 2022?",
     "15", "sql"),

    ("Who scored the most runs in IPL 2024?",
     "Kohli", "sql"),

    ("Explain Kohli batting performance in 2022",
     "341", "vector"),

    ("Explain Bumrah bowling performance and total wickets in 2022",
     "15", "hybrid")
]

results = []

# Evaluate RAG pipeline
for question, expected, expected_intent in test_data:

    start = time.perf_counter()

    result = generate_rag_response(question)

    latency = time.perf_counter() - start

    answer = result["answer"] or ""
    context = result["context"] or ""

    pattern = r"\b" + re.escape(expected.lower()) + r"\b"

    results.append({
        "Question": question,
        "Expected": expected,
        "Generated Answer": answer,
        "Intent": result["intent"],
        "Intent Correct": result["intent"] == expected_intent,
        "Answer Match": bool(re.search(pattern, answer.lower())),
        "Context Match": bool(re.search(pattern, context.lower())),
        "Model": result["model"],
        "Latency": round(latency, 2)
    })

# Evaluation results
evaluation_df = pd.DataFrame(results)

display(evaluation_df)

# Overall evaluation summary
print("\n========== RAG EVALUATION ==========")

print("Total Questions:", len(evaluation_df))

print("Answer Match Accuracy:",
      round(evaluation_df["Answer Match"].mean() * 100, 2), "%")

print("Intent Accuracy:",
      round(evaluation_df["Intent Correct"].mean() * 100, 2), "%")

print("Context Match Rate:",
      round(evaluation_df["Context Match"].mean() * 100, 2), "%")

print("Average Response Time:",
      round(evaluation_df["Latency"].mean(), 2), "seconds")

print("API Success Rate:",
      round(evaluation_df["Model"].notna().mean() * 100, 2), "%")


Extracted Information: {'question': 'How many runs did Kohli score in IPL 2022?', 'player': 'V Kohli', 'season': '2022', 'category': 'batting', 'intent': 'sql'}
gemini-2.5-flash unavailable: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}. Trying next model...
gemini-2.5-pro unavailable: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use models/gemini-3.1-pro-preview for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}. Trying next model...
Extracted Information: {'question': 'How many wickets d

,Question,Expected,Generated Answer,Intent,Intent Correct,Answer Match,Context Match,Model,Latency
0,How many runs did Kohli score in IPL 2022?,341,V Kohli scored **341** runs in IPL 2022.,sql,True,True,True,gemini-3.5-flash,2.47
1,How many wickets did Bumrah take in IPL 2022?,15,"Based on the provided SQL statistics, JJ Bumra...",sql,True,True,True,gemini-3.5-flash,2.73
2,Who scored the most runs in IPL 2024?,Kohli,"V Kohli scored the most runs in IPL 2024, with...",sql,True,True,True,gemini-3.5-flash,2.54
3,Explain Kohli batting performance in 2022,341,Based on the provided data for the 2022 IPL se...,vector,True,True,True,gemini-3.5-flash,8.52
4,Explain Bumrah bowling performance and total w...,15,"In the 2022 IPL season, JJ Bumrah's bowling pe...",hybrid,True,True,True,gemini-3.5-flash,4.16



========== RAG EVALUATION ==========
Total Questions: 5
Answer Match Accuracy: 100.0 %
Intent Accuracy: 100.0 %
Context Match Rate: 100.0 %
Average Response Time: 4.08 seconds
API Success Rate: 100.0 %


## FastAPI Backend Development

### Save databases to Google Drive

In [87]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

backup_path = "/content/drive/MyDrive/IPL_RAG"

os.makedirs(backup_path, exist_ok=True)

# Close SQLite connection
conn.close()

# Save SQLite database
shutil.copy2(
    "/content/ipl_database/ipl_stats.db",
    backup_path + "/ipl_stats.db"
)

# Release ChromaDB references
del collection
del chroma_client

# Save ChromaDB database
shutil.make_archive(
    backup_path + "/ipl_chromadb",
    "zip",
    "/content/ipl_chromadb"
)

print("Backup completed successfully!")

print("\nSaved Files:")

for file in os.listdir(backup_path):
    print(file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backup completed successfully!

Saved Files:
ipl_stats.db
ipl_chromadb.zip


## Install Gradio in Colab

In [88]:
!pip install -q gradio

In [89]:
import gradio as gr

print(gr.__version__)

6.31.0


### Create chatbot interface


In [91]:
import sqlite3
import chromadb

# Restore SQLite connection
conn = sqlite3.connect(
    "/content/ipl_database/ipl_stats.db",
    check_same_thread=False
)

# Restore ChromaDB connection
chroma_client = chromadb.PersistentClient(
    path="/content/ipl_chromadb"
)

collection = chroma_client.get_collection(
    name="ipl_cricket"
)

print("SQLite Connected Successfully")
print("ChromaDB Documents:", collection.count())

SQLite Connected Successfully
ChromaDB Documents: 5162


In [93]:
import gradio as gr
import time
import traceback

# Conversational memory
def rewrite_question(message, history):

    if not history:
        return message

    prompt = f"""
    Rewrite the current question as a standalone IPL question.

    Conversation history:
    {history[-6:]}

    Current question:
    {message}

    Resolve player names, seasons and references.
    Preserve all players and seasons mentioned.
    Do not invent information.
    Return only the rewritten question.
    """

    result = generate_with_fallback(prompt)

    if result.get("model") and result.get("answer"):
        return result["answer"].strip()

    return message


# Connect existing RAG engine
def chat_response(message, history):

    start = time.perf_counter()

    try:
        question = rewrite_question(message, history)

        result = generate_rag_response(question)

        latency = round(time.perf_counter() - start, 2)

        return f"""
{result['answer']}

---

**Retrieval Mode:** {result['intent'].upper()}

**AI Model:** {result['model']}

**Response Time:** {latency} seconds
"""

    except Exception:
        traceback.print_exc()
        return "Unable to generate a response. Please try again."


# Close previous Gradio application
gr.close_all()

# UI customization
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="orange",
    neutral_hue="slate"
)

# Chatbot interface
demo = gr.ChatInterface(
    fn=chat_response,

    title="🏏 IPL Cricket Intelligence RAG",

    description="""
    ### Your AI-Powered IPL Cricket Analyst

    Explore player statistics, batting and bowling
    performance, and historical IPL records.

    **Hybrid RAG | SQLite | ChromaDB | BGE | Gemini**
    """,

    examples=[
        "How many runs did Kohli score in IPL 2022?",
        "How many wickets did Bumrah take in IPL 2022?",
        "Who scored the most runs in IPL 2024?",
        "Explain Kohli batting performance in 2024",
        "Explain Bumrah bowling performance in 2024"
    ]
)

# Launch application
demo.launch(
    share=True,
    debug=True,
    theme=theme
)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4dd7d660858e866792.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Original Question: How many runs did Kohli score in IPL 2022?
Rewritten Question: How many runs did Kohli score in IPL 2022?
Extracted Information: {'question': 'How many runs did Kohli score in IPL 2022?', 'player': 'V Kohli', 'season': '2022', 'category': 'batting', 'intent': 'sql'}
Original Question: ms dhoni overall ipl runs
Rewritten Question: How many overall runs has MS Dhoni scored in the IPL?
Extracted Information: {'question': 'How many overall runs has MS Dhoni scored in the IPL?', 'player': 'MS Dhoni', 'season': None, 'category': 'batting', 'intent': 'sql'}
Original Question: Compare Virat Kohli and Rohit Sharma's batting performance in IPL 2024
Rewritten Question: Compare Virat Kohli and Rohit Sharma's batting performance in IPL 2024.
Extracted Information: {'question': "Compare Virat Kohli and Rohit Sharma's batting performance in IPL 2024.", 'player': 'RG Sharma', 'season': '2024', 'category': 'batting', 'intent': 'hybrid'}
Original Question: Compare Virat Kohli and Rohi